In [ ]:
!pip install --upgrade scikit-learn xgboost==3.3.0 lightgbm==4.7.0 catboost==1.2.10

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pandas.api.types import CategoricalDtype
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, LabelEncoder, label_binarize, OrdinalEncoder, QuantileTransformer, TargetEncoder, RobustScaler, FunctionTransformer, KBinsDiscretizer
from category_encoders import CatBoostEncoder, MEstimateEncoder

from sklearn.ensemble import RandomForestClassifier, VotingClassifier, HistGradientBoostingClassifier, GradientBoostingClassifier, HistGradientBoostingRegressor, AdaBoostRegressor, RandomForestRegressor, ExtraTreesRegressor, ExtraTreesClassifier
from sklearn.linear_model import RidgeClassifier, LogisticRegression, LinearRegression, BayesianRidge, Ridge, ElasticNet, Lasso
from sklearn import set_config
from sklearn.decomposition import PCA
import umap
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import optuna
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve, root_mean_squared_error, mean_squared_error, precision_recall_curve, make_scorer, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay, matthews_corrcoef, r2_score, balanced_accuracy_score, average_precision_score
from scipy.stats import norm, skew
from scipy.special import expit, logit 

from colorama import Fore, Style, init
from copy import deepcopy
from sklearn.base import BaseEstimator, TransformerMixin
from pprint import pprint
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, StratifiedKFold, KFold, RepeatedKFold, cross_val_score, StratifiedGroupKFold
from sklearn.isotonic import IsotonicRegression
from xgboost import DMatrix, XGBClassifier, XGBRegressor
import xgboost as xgb
from lightgbm import log_evaluation, early_stopping, LGBMClassifier, LGBMRegressor, Dataset
import lightgbm
from catboost import CatBoostClassifier, CatBoostRegressor, Pool
from tqdm.notebook import tqdm
from optuna.samplers import TPESampler, CmaEsSampler
from optuna.pruners import HyperbandPruner
from functools import partial
from IPython.display import display_html, clear_output
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.compose import ColumnTransformer
import gc
import re
from typing import Literal, NamedTuple
from itertools import combinations
from matplotlib.colors import LinearSegmentedColormap
from sklearn.manifold import TSNE
from matplotlib.lines import Line2D
from sklearn.inspection import permutation_importance
from scipy.optimize import minimize, differential_evolution

import keras
from keras.models import Sequential
from keras import layers
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import regularizers

import math
import random
from copy import deepcopy
from typing import Any, Literal, NamedTuple, Optional
import torch

# <p style="border-radius: 40px; color: white; font-weight: bold; font-size: 150%; text-align: center; background-color:#3cb371; padding: 5px 5px 5px 5px;">Configuration</p>

In [ ]:
class Config:
    target = 'Will_Buy_EV'
    train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e9/train.csv', index_col='id')
    test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e9/test.csv', index_col='id')
    submission = pd.read_csv('/kaggle/input/competitions/playground-series-s6e9/sample_submission.csv')
    orig = pd.read_csv('/kaggle/input/datasets/itzzomkar/ev-adoption-behavior-and-range-anxiety/EV_Adoption_and_Range_Anxiety_Dataset.csv', index_col='Buyer_ID')[train.columns]

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    state = 42
    n_splits = 10
    early_stop = 500
    metric = 'roc_auc'
    task_type = "binary"
    task_is_regression = task_type == 'regression'

    mapping = {"No": 0, "Yes": 1}
    inv_mapping = {v: k for k, v in mapping.items()}
    train[target] = train[target].map(mapping)
    orig[target] = orig[target].map(mapping)

    if task_is_regression:
        n_classes = 1
        folds = KFold(n_splits=n_splits, shuffle=True, random_state=state)
    else:
        n_classes = train[target].nunique()
        labels = list(train[target].unique())
        folds = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=state)   

# <p style="border-radius: 40px; color: white; font-weight: bold; font-size: 150%; text-align: center; background-color:#3cb371; padding: 5px 5px 5px 5px;">EDA</p>

In [ ]:
class EDA(Config):
    
    def __init__(self):
        super().__init__()

        self.cat_features = self.train.drop(self.target, axis=1).select_dtypes(include=['object', 'bool']).columns.tolist()
        self.num_features = self.train.drop(self.target, axis=1).select_dtypes(exclude=['object', 'bool']).columns.tolist()
        self.data_info()
        self.heatmap()
        self.dist_plots()
        self.cat_feature_plots()
        self.pairplot()
        if self.task_is_regression:
            self.target_plot()
        else:
            self.target_pie()
                
    def data_info(self):        
        for data, label in zip([self.train, self.test], ['Train', 'Test']):
            table_style = [{'selector': 'th:not(.index_name)',
                            'props': [('background-color', '#3cb371'),
                                      ('color', '#FFFFFF'),
                                      ('font-weight', 'bold'),
                                      ('border', '1px solid #DCDCDC'),
                                      ('text-align', 'center')]
                            }, 
                            {'selector': 'tbody td',
                             'props': [('border', '1px solid #DCDCDC'),
                                       ('font-weight', 'normal')]
                            }]
            print(Style.BRIGHT+Fore.GREEN+f'\n{label} head\n')
            display(data.head().style.set_table_styles(table_style))
                           
            print(Style.BRIGHT+Fore.GREEN+f'\n{label} info\n'+Style.RESET_ALL)               
            display(data.info())
                           
            print(Style.BRIGHT+Fore.GREEN+f'\n{label} describe\n')
            display(data.describe().drop(index='count', columns=self.target, errors = 'ignore').T
                    .style.set_table_styles(table_style).format('{:.3f}'))

            miss = data.isna().sum()
            miss = miss[miss > 0].sort_values(ascending=False)
            if miss.empty:
                print(Style.BRIGHT + Fore.GREEN + f'\nThere are no missing values in the {label}\n' + Style.RESET_ALL)
            else:
                df_train = pd.DataFrame({
                    'missing_count': miss,
                    'missing_frac': (miss / len(data)).round(4)
                })
                print(Style.BRIGHT + Fore.GREEN + f'\n{label} missing values\n' + Style.RESET_ALL)
                display(pd.DataFrame({
                    'missing_count': miss,
                    'missing_frac': (miss / len(data)).round(4)
                }))
    
    def heatmap(self):
        n = len(self.num_features) + 1
        cell = 0.7        
        
        print(Style.BRIGHT+Fore.GREEN+f'\nCorrelation Heatmap\n')
        plt.figure(figsize=(n * cell, n * cell))
        corr = self.train[self.num_features+[self.target]].corr(method='spearman')
        sns.heatmap(corr, fmt = '0.2f', cmap = 'Greens', square=True, annot=True, linewidths=1, cbar=False)
        plt.savefig('1.png', dpi=300, bbox_inches='tight')
        plt.show()
        
    def dist_plots(self):
        print(Style.BRIGHT+Fore.GREEN+f"\nDistribution analysis\n")
        df = pd.concat([self.train[self.num_features].assign(Source='Train'),
                        self.test[self.num_features].assign(Source='Test')],
                        axis=0, ignore_index=True)

        n = len(self.num_features)
        if n == 0:
            return
        fig, axes = plt.subplots(n, 2, figsize=(18, n * 4.5),
                                 gridspec_kw={'hspace': 0.3, 'wspace': 0.2, 'width_ratios': [0.70, 0.30]})
        if n == 1:
            axes = np.array([axes])

        for i, col in enumerate(self.num_features):
            ax = axes[i, 0]
            sns.kdeplot(data=df, x=col, hue='Source', palette=['#3cb371', '#ef5350'], ax=ax, linewidth=2)
            med_train = self.train[col].median()
            med_test = self.test[col].median()
            ax.axvline(med_train, color='#3cb371', linestyle='--', linewidth=1)
            ax.axvline(med_test, color='#ef5350', linestyle='--', linewidth=1)
            ax.set_title(f"{col}  KDE")
            ax.grid()
            ax.text(0.98, 0.95, f"skew={self.train[col].skew():.2f}\nmed={med_train:.2f}", transform=ax.transAxes, ha='right', va='top', fontsize=9, bbox=dict(alpha=0.2))
            
            ax2 = axes[i, 1]
            sns.boxplot(data=df, y=col, x='Source', width=0.5, linewidth=1, fliersize=1, ax=ax2, palette=['#3cb371', '#ef5350'])
            ax2.set_title(f"{col}  Boxplot")
            ax2.set_xticklabels(['Train', 'Test'])
            ax2.tick_params(axis='both', which='major')
            plt.savefig('3.png', dpi=300, bbox_inches='tight')
            
        plt.tight_layout()
        plt.show()
               
    def cat_feature_plots(self, top_k: int = 10):
        print(Style.BRIGHT+Fore.GREEN+f"\nCategorical feature analysis\n")
        m = max(len(self.cat_features), 1)
        fig, axes = plt.subplots(m, 2, figsize=(18, m * 4.5), gridspec_kw={'hspace': 0.5, 'wspace': 0.2})
        if len(self.cat_features) == 0:
            return
        if len(self.cat_features) == 1:
            axes = np.array([axes])

        for i, col in enumerate(self.cat_features):
            ax = axes[i, 0]
            vc = self.train[col].value_counts().nlargest(top_k)
            vc_pct = (vc / len(self.train) * 100).round(2)
            sns.barplot(x=vc.index.astype(str), y=vc.values, ax=ax, color='#3cb371')
            for j, v in enumerate(vc.values):
                ax.text(j, v + max(vc.values) * 0.01, f"{v} ({vc_pct.iloc[j]}%)", ha='center', fontsize=9)
            ax.set_title(f"{col} Train top {top_k}")
            ax.set_xticklabels(ax.get_xticklabels(), ha='right')

            ax2 = axes[i, 1]
            vc_t = self.test[col].value_counts().nlargest(top_k)
            vc_t_pct = (vc_t / len(self.test) * 100).round(2)
            sns.barplot(x=vc_t.index.astype(str), y=vc_t.values, ax=ax2, color='#ef5350')
            for j, v in enumerate(vc_t.values):
                ax2.text(j, v + max(vc_t.values) * 0.01, f"{v} ({vc_t_pct.iloc[j]}%)", ha='center', fontsize=9)
            ax2.set_title(f"{col} Test top {top_k}")
            ax2.set_xticklabels(ax2.get_xticklabels(), ha='right')

        plt.tight_layout()
        plt.show()

    def pairplot(self, n=800):
        if len(self.num_features) >= 2:
            print(Style.BRIGHT+Fore.GREEN+f'\nPair plot\n')
            sample = self.train.assign(Source='Train').sample(n=n, random_state=42)
            test_sample = self.test.assign(Source='Test').sample(n=min(n, len(self.test)), random_state=42)
            sample = pd.concat([sample, test_sample], ignore_index=True)
            
            sns.pairplot(
                sample[self.num_features + ['Source']],
                hue='Source',
                diag_kind='kde',
                corner=True,
                palette={'Train': '#3cb371', 'Test': '#ef5350'},
                plot_kws={'alpha': 0.6, 's': 12},
                diag_kws={'linewidth': 2, 'fill': True}
            )
            plt.savefig('4.png', dpi=300, bbox_inches='tight')
            plt.show()
                
    def target_pie(self):
        print(Style.BRIGHT+Fore.GREEN+f"\nTarget feature distribution\n")
        plt.figure(figsize=(6, 6))
        counts = (self.train[self.target].value_counts(normalize=True).reindex(list(range(self.train[self.target].nunique(dropna=True))), fill_value=0))
        plt.pie(counts.values, labels=self.labels, autopct='%1.2f%%', colors=['#3cb371', '#ef5350', '#5c9ded'])
        plt.show() 

    def target_plot(self):
        print(Style.BRIGHT+Fore.GREEN+f"\nTarget feature distribution\n")
        fig, axes = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={'hspace': 0.3, 'wspace': 0.2, 'width_ratios': [0.70, 0.30]})
        ax = axes[0]
        sns.kdeplot(data=self.train, x=self.target, color='#3cb371', ax=ax, linewidth=2)
        ax.set_title(f"{self.target} KDE")
        ax.grid()

        ax2 = axes[1]
        sns.boxplot(data=self.train, y=self.target, width=0.5, linewidth=1, fliersize=1, ax=ax2, color='#3cb371')
        ax2.set_title(f"{self.target} Boxplot")
        plt.tight_layout()
        plt.show()

In [ ]:
eda = EDA()

### Key correlations

- **Charging_Stations_Near_Home ↔ Charging_Stations_Near_Work (≈ 0.54)**  
  Moderate positive correlation: areas with home charging infrastructure tend to also have workplace charging available.

- **Environmental_Concern_Level ↔ Will_Buy_EV (≈ 0.46)**  
  Moderate positive correlation: higher environmental concern is associated with greater willingness to purchase an EV.

- **Charging_Stations_Near_Home ↔ Will_Buy_EV (positive, notable)**  
  Proximity of charging infrastructure near home increases the likelihood of intending to buy an EV.

- **Daily_Commute_km ↔ Will_Buy_EV (context‑dependent)**  
  If positive, longer commutes increase EV purchase intent (economic/efficiency argument); if weak, commute length is not a universal driver.

- **Age / Annual_Income_USD / Number_of_Cars_Owned ↔ Will_Buy_EV (weaker or mixed)**  
  Demographic signals are less consistent predictors compared with infrastructure and environmental concern.

---

### Interpretation

- **Infrastructure as an enabler:** Availability of charging stations is a practical, actionable factor that lowers adoption barriers. Investing in or partnering to expand local charging networks will likely increase EV uptake in targeted regions.

- **Environmental motivation as a targeting signal:** Consumers with higher environmental concern form a natural target segment; messaging focused on emissions reduction and sustainability will resonate with them.

### Analysis of Train vs Test distributions

**Distribution alignment:** For key features the annotated medians match expected values: **age med = 47**, **Annual_Income_USD med = 84,880**, **Daily_Commute_km med = 33.6**, **Number_of_Cars_Owned med = 2**, **Charging_Stations_Near_Home med = 4**, **Charging_Stations_Near_Work med = 6**. KDEs and boxplots show good agreement between Train and Test — no obvious median drift is observed.

**Skewness:** Reported skew values indicate most features are near symmetric or mildly skewed: **age skew ≈ 0.00**, **income skew ≈ −0.01**, **commute skew ≈ −0.08**, **cars skew ≈ −0.81**, **charging_home skew ≈ −0.70**, **charging_work skew ≈ 0.70**. Negative skew for number of cars and home charging suggests concentration above the median; positive skew for work charging indicates a long right tail.

**Tails and outliers:** Boxplots reveal outliers and long tails, especially for infrastructure‑related features (charging stations) and, possibly, income; this is visible in the whiskers and individual points.

**Conclusion:** Train and Test are broadly comparable in central tendency, but asymmetries and heavy tails are present and may affect model training and evaluation.

# <p style="border-radius: 40px; color: white; font-weight: bold; font-size: 150%; text-align: center; background-color:#3cb371; padding: 5px 5px 5px 5px;">Preprocessing</p>

In [ ]:
class Preprocessing:
    def __init__(self, target):
        self.target = target
        self._fitted = False

    def fit_transform(self, X_train: pd.DataFrame, y_train: pd.Series, orig: pd.DataFrame = None):
        self.fit(X_train, y_train, orig=orig)
        return self.transform(X_train)

    def fit(self, X_train: pd.DataFrame, y_train: pd.Series, orig: pd.DataFrame = None):
        self.X_train_ = X_train.copy()
        self.y_train_ = y_train.copy()

        self.num_features_ = self.X_train_.select_dtypes(exclude=['object', 'bool', 'category']).columns.tolist()
        self.cat_features_ = self.X_train_.select_dtypes(include=['object', 'bool', 'category']).columns.tolist()
        
        self._orig_stats = {}

        if orig is not None:
            self.orig_ = orig.copy()
            self._fit_orig_target_stats(self.cat_features_ + self.num_features_)

        self._fit_feature_engineering(self.X_train_)
        X_fe_train = self._apply_feature_engineering(self.X_train_)

        self._fit_frequency_encoding(X_fe_train)
        X_fe_train = self._apply_frequency_encoding(X_fe_train)
        
        X_fe_train = self._finalize_types(X_fe_train, fit=True)

        self._fitted = True
        return self

    def transform(self, X: pd.DataFrame):
        if not self._fitted:
            raise RuntimeError("Preprocessing is not fitted. Call fit() first.")

        X = X.copy()

        X = self._apply_orig_target_stats(X)

        X = self._apply_feature_engineering(X)

        X = self._apply_frequency_encoding(X)

        X = self._finalize_types(X, fit=False)

        for c in self.cat_features:
            X[c] = X[c].astype(object).fillna('NaN').astype('category')

        return X, self.cat_features, self.num_features

    def _fit_orig_target_stats(self, cols):
        self._orig_global_mean = float(self.orig_[self.target].mean())
        for c in cols:
            if c in self.orig_.columns:
                self._orig_stats[c] = self.orig_.groupby(c, observed=False)[self.target].mean()
            else:
                self._orig_stats[c] = pd.Series(dtype=float)

    def _apply_orig_target_stats(self, df):
        df = df.copy()
        for c, ser in self._orig_stats.items():
            col = f"{c}_org_mean"
            if c in df.columns and not ser.empty:
                df[col] = df[c].map(ser).astype(float).fillna(self._orig_global_mean)
            else:
                df[col] = self._orig_global_mean
        return df

    def _fit_feature_engineering(self, X_train):
        X_train = X_train.copy()

        self._highcard_num = [c for c in self.num_features_ if X_train[c].nunique(dropna=False) > 20]
        self.category_map = {}

        bin_config = {'Annual_Income_USD': [400, 600, 800, 900, 1100]}
        for col, bins_list in bin_config.items():
            for n_bins in bins_list:
                bin_name = f"{col}_bin_{n_bins}"
                kb = KBinsDiscretizer(n_bins=n_bins, encode='ordinal', strategy='quantile', subsample=None)
                kb.fit_transform(X_train[[col]]).ravel().astype('int32')
                self.category_map[bin_name] = kb
                
    def _apply_feature_engineering(self, df):
        df = df.copy()

        self.num_to_cat = []
        for col in self.num_features_:
            cat_name = f"{col}_cat"
            df[cat_name] = df[col].fillna('NaN').astype(str)
            self.num_to_cat.append(cat_name)

        bin_config = {'Annual_Income_USD': [400, 600, 800, 900, 1100]}
        for col, bins_list in bin_config.items():
            for n_bins in bins_list:
                bin_name = f"{col}_bin_{n_bins}"
                kb = self.category_map[bin_name]
                df[bin_name] = kb.transform(df[[col]]).ravel().astype('int32')
                df[bin_name] = df[bin_name].astype('category')
                
        for c in self.num_features_:
            for k in range(-4,4):
                df[f"{c}_digit{k}"] = (df[c].fillna(0) // (10**k) % 10).astype('int8')

        df['index'] = df['Environmental_Concern_Level'] * df['Charging_Stations_Near_Home']
        df['is_30k_spike'] = (df['Annual_Income_USD'] == 30000.0).astype('int8')
        df['is_millionaire_cliff'] = (df['Annual_Income_USD'] >= 169972).astype('int8')
        df['is_dead_zone'] = ((df['Annual_Income_USD'] >= 31003 ) & (df['Annual_Income_USD'] <= 41970)).astype('int8')      
        df['is_env_hater'] = (df['Environmental_Concern_Level'] == 1).astype('int8')
        df['income100_floor']  = np.floor(df['Annual_Income_USD'] / 100.0).astype(str)
        df['income1000_floor'] = np.floor(df['Annual_Income_USD'] / 1000.0).astype(str)
        df['commute_integer']  = np.floor(df['Daily_Commute_km']).astype(str)    

        return df

    def _fit_frequency_encoding(self, X):
        all_cats = list(dict.fromkeys(self.cat_features_+self.num_to_cat))
        self._fe_cols = [c for c in all_cats if c in X.columns]

        self._freq_encodings = {}
        for c in self._fe_cols:
            self._freq_encodings[c] = X[c].value_counts(normalize=True).to_dict()

    def _apply_frequency_encoding(self, X):
        X = X.copy()
        for c in getattr(self, "_fe_cols", []):
            mapping = self._freq_encodings.get(c, {})
            if c in X.columns:
                X[f"{c}_fe"] = X[c].map(mapping).astype(float).fillna(0.0)
            else:
                X[f"{c}_fe"] = 0.0
        return X
        
    def _finalize_types(self, X, fit=False):
        self.num_features = X.select_dtypes(exclude=['object', 'bool', 'category']).columns.tolist()
        self.cat_features = X.select_dtypes(include=['object', 'bool', 'category']).columns.tolist()
        return X

In [ ]:
y = Config.train[Config.target]
X = Config.train.drop(Config.target, axis=1)

p = Preprocessing(Config.target)
X, cat_features, num_features = p.fit_transform(X, y)
test, _, _ = p.transform(Config.test)

# <p style="border-radius: 40px; color: white; font-weight: bold; font-size: 150%; text-align: center; background-color:#3cb371; padding: 5px 5px 5px 5px;">Models</p>

In [ ]:
models = {
    'XGB2_29': XGBClassifier(**{'n_estimators': 50000,
                               'learning_rate': 0.01,
                               'random_state': Config.state,
                               'objective': 'binary:logistic',
                               'eval_metric': 'auc',
                               'enable_categorical': True,
                               'device': 'cuda',
                               'early_stopping_rounds': Config.early_stop,
                               'max_bin': 5000,
                               'lambda': 9.979259191043475,
                               'alpha': 1.4214099968896257, 
                               'colsample_bytree': 0.713288950454839,
                               'subsample': 0.9561912985547617,
                               'max_depth': 5, 
                               'min_child_weight': 2
                              }),
}

# <p style="border-radius: 40px; color: white; font-weight: bold; font-size: 150%; text-align: center; background-color:#3cb371; padding: 5px 5px 5px 5px;">Training</p>

In [ ]:
class FeatureEncoder:
    def __init__(self, num_features, cat_features):
        self.num_features = num_features
        self.cat_features = cat_features
        self.ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
        self.scaler = StandardScaler()
        self.ohe_cols = None

    def fit(self, X):
        self.ohe.fit(X[self.cat_features])
        self.ohe_cols = self.ohe.get_feature_names_out(self.cat_features)
        self.scaler.fit(X[self.num_features])
        
    def transform_fold(self, X_train, X_val, X_test):
        def transform(X):
            X[self.num_features] = self.scaler.transform(X[self.num_features])

            X_ohe = self.ohe.transform(X[self.cat_features])
            X_ohe_df = pd.DataFrame(X_ohe, columns=self.ohe_cols, index=X.index)   
            X = pd.concat([X.drop(columns=self.cat_features).reset_index(drop=True),
                                        X_ohe_df.reset_index(drop=True)], axis=1)
            return X
        return transform(X_train), transform(X_val), transform(X_test)

In [ ]:
class Metrics:
    def __init__(self, task_type: str, metric: str, n_classes=None):
        self.task_type = task_type
        self.metric = metric
        self.n_classes = n_classes

    def _to_labels(self, y_pred):
        if self.task_type == "multiclass":
            return np.argmax(y_pred, axis=1)
        if self.task_type == "binary":
            return (y_pred >= 0.5).astype(int)
        return y_pred

    def score(self, y_true, y_pred):
        if self.metric == "roc_auc":
            if self.task_type == "multiclass":
                return roc_auc_score(y_true, y_pred, multi_class="ovr")
            return roc_auc_score(y_true, y_pred)
        if self.task_type == "regression":
            if self.metric == "mae":
                return mean_absolute_error(y_true, y_pred)
            if self.metric == "r2":
                return r2_score(y_true, y_pred)
            if self.metric == "rmse" or self.metric == "rmsle":
                return root_mean_squared_error(y_true, y_pred)
            if self.metric == "mse":
                return mean_squared_error(y_true, y_pred, squared=True)
                
        y_label = self._to_labels(y_pred)
        if self.metric == "accuracy":
            return accuracy_score(y_true, y_label)
        if self.metric == "balanced_accuracy":
            return balanced_accuracy_score(y_true, y_label)
        if self.metric == "f1":
            avg = "weighted" if self.n_classes and self.n_classes > 2 else "binary"
            return f1_score(y_true, y_label, average=avg)
        if self.metric == "precision":
            avg = "weighted" if self.n_classes and self.n_classes > 2 else "binary"
            return precision_score(y_true, y_label, average=avg, zero_division=0)
        if self.metric == "recall":
            avg = "weighted" if self.n_classes and self.n_classes > 2 else "binary"
            return recall_score(y_true, y_label, average=avg, zero_division=0)
        raise ValueError(f"Unsupported metric: {self.metric}")

In [ ]:
class Plotter(Config):       
    def plot_result(self, y_true, preds):
        if self.task_is_regression:
            return self.plot_regression(y_true, preds)
        return self.plot_classification(y_true, preds)

    def score_bar(self, scores_df):
        plt.figure(figsize=(16, max(6, len(scores_df) * 0.2)))
        colors = ['#3cb371' if i != 'Ensemble' else '#ef5350' for i in scores_df.index]
        hbars = plt.barh(scores_df.index, scores_df.Score, color=colors, height=0.6)
        plt.xlim(0.95, 0.975)
        plt.bar_label(hbars, fmt='%.6f')
        plt.tight_layout()
        plt.show()

    def plot_classification(self, y_true, preds):
        if self.n_classes == 2:
            y_bin = pd.DataFrame({
                self.labels[0]: (y_true == 0).astype(int),
                self.labels[1]: (y_true == 1).astype(int)
            })
        
            probs = np.zeros((len(y_true), self.n_classes))
            probs[:, 1] = preds
            probs[:, 0] = 1 - preds
            y_pred = (probs[:, 1] >= 0.5).astype(int)
        else:
            y_bin = pd.DataFrame(label_binarize(y_true, classes=range(self.n_classes)), columns=self.labels)
            probs = preds
            y_pred = np.argmax(probs, axis=1)
        
        fig, axes = plt.subplots(2, 2, figsize=(14, 12))
        ax1, ax2, ax3, ax4 = axes.ravel()
        colors = ['#3cb371', '#ef5350', '#5c9ded', '#ffa726', '#ab47bc']
        
        for i, name in enumerate(self.labels):
            RocCurveDisplay.from_predictions(
                y_bin.iloc[:, i],
                probs[:, i],
                name=name,
                ax=ax1
            )
            ax1.lines[-1].set_color(colors[i])
            
        ax1.plot([0, 1], [0, 1], '--', color='black')
        ax1.set_title('ROC (one-vs-rest)')
        ax1.set_xlabel('False Positive Rate')
        ax1.set_ylabel('True Positive Rate')
        ax1.legend(loc="lower right")
        
        ConfusionMatrixDisplay.from_predictions(
            y, y_pred,
            display_labels=self.labels,
            colorbar=False,
            ax=ax2,
            cmap='Greens'
        )
        ax2.set_title('Confusion Matrix')
        
        for i, name in enumerate(self.labels):
            precision, recall, _ = precision_recall_curve(y_bin.iloc[:, i], probs[:, i])
            ap = average_precision_score(y_bin.iloc[:, i], probs[:, i])
            ax3.plot(recall, precision, label=f"{name} (AP={ap:.3f})", color=colors[i])
        
        ax3.set_xlabel("Recall")
        ax3.set_ylabel("Precision")
        ax3.set_title("Precision–Recall Curves (One-vs-Rest)")
        ax3.legend(loc='best')
        
        bins = np.linspace(0, 1, 20)
        
        for cls, color in zip(range(self.n_classes), colors):
            p = probs[:, cls]
            is_cls = y_bin.iloc[:, cls].values
        
            bin_centers, bin_pos = [], []
            for i in range(len(bins) - 1):
                mask = (p >= bins[i]) & (p < bins[i + 1])
                if mask.sum() > 0:
                    bin_centers.append((bins[i] + bins[i + 1]) / 2)
                    bin_pos.append(is_cls[mask].mean())
        
            ax4.plot(bin_centers, bin_pos, 'o-', color=color, label=self.labels[cls])
        
        ax4.plot([0, 1], [0, 1], '--', color='black')
        ax4.set_title('Calibration Curves (All Classes)')
        ax4.set_xlabel('Predicted probability')
        ax4.set_ylabel('Fraction positive')
        ax4.legend()
        plt.savefig('8.png', dpi=300, bbox_inches='tight')
        
        plt.tight_layout()
        plt.show()
    
    def plot_regression(self, y_true, preds):
        resid = y_true - preds
        abs_err = np.abs(resid)
    
        cmap = LinearSegmentedColormap.from_list("g2r", ["#3cb371", "#ef5350"])
        norm = plt.Normalize(vmin=abs_err.min(), vmax=abs_err.max())
        colors = cmap(norm(abs_err))
    
        fig, axes = plt.subplots(2, 2, figsize=(14, 12))
        ax1, ax2, ax3, ax4 = axes.ravel()
    
        sc = ax1.scatter(y_true, preds, c=colors, s=30, alpha=0.9)
        mn = min(y_true.min(), preds.min())
        mx = max(y_true.max(), preds.max())
        ax1.plot([mn, mx], [mn, mx], 'k--', lw=1)
        ax1.set_xlabel('Actual')
        ax1.set_ylabel('Predicted')
        ax1.set_title('Actual vs Predicted (OOF)')
    
        ax2.scatter(preds, resid, c=colors, s=30, alpha=0.9, edgecolor='none')
        ax2.axhline(0, color='k', linestyle='--', lw=1)
        ax2.set_xlabel('Predicted')
        ax2.set_ylabel('Residual (Actual - Predicted)')
        ax2.set_title('Residuals vs Predicted (OOF)')
    
        sns.histplot(resid, bins=100, kde=True, color='#3cb371', ax=ax3)
        ax3.set_xlabel('Error')
        ax3.set_title('Error distribution')

        (theoretical_q, ordered_vals), (slope, intercept, r) = probplot(resid, dist="norm")
        colors_ordered = cmap(norm(abs_err[np.argsort(resid)]))        
        ax4.scatter(theoretical_q, ordered_vals, c=colors_ordered, s=30, edgecolor='none', alpha=0.9)
        x_line = np.array([theoretical_q.min(), theoretical_q.max()])
        y_line = intercept + slope * x_line
        ax4.plot(x_line, y_line, 'k--', lw=1)
        ax4.set_xlabel('Theoretical quantiles (normal)')
        ax4.set_ylabel('Ordered residuals (OOF)')
        ax4.set_title('Q-Q plot (OOF residuals)')
        
        plt.tight_layout()
        plt.show()

In [ ]:
class Trainer(Config):
    
    def __init__(self, X, y, test, models, num_features, cat_features, training=True):
        self.X = X
        self.test = test
        self.y = y
        self.models = models
        self.training = training
        self.scores = pd.DataFrame(columns=['Score'], dtype=float)
        self.OOF_preds = pd.DataFrame(dtype=float)
        self.TEST_preds = pd.DataFrame(dtype=float)
        self.num_features = num_features
        self.cat_features = cat_features
        self.metrics = Metrics(task_type=self.task_type, metric=self.metric, n_classes=self.n_classes)
        self.plotter = Plotter()

    def train(self, model, X, y, test, model_name):
        if self.task_type == "multiclass":
            oof_pred = np.zeros((X.shape[0], self.n_classes), dtype=float)
            test_pred = np.zeros((test.shape[0], self.n_classes), dtype=float)
        else:
            oof_pred = np.zeros(X.shape[0], dtype=float)
            test_pred = np.zeros(test.shape[0], dtype=float)

        print('='*20)
        print(model_name)
        
        for n_fold, (train_id, valid_id) in enumerate(self.folds.split(X, y)):
            X_train = X.iloc[train_id].copy()
            y_train = y.iloc[train_id]
            X_val = X.iloc[valid_id].copy()
            y_val = y.iloc[valid_id]
            X_test = test.copy()
            
            if model_name != 'Ensemble':                 
                te_cols = self.cat_features+ [c for c in X_train.columns if '_digit' in c]
                
                for sm in [10, 100, 'auto']:
                    te = TargetEncoder(smooth=sm, random_state=42)
                
                    X_train_enc = te.fit_transform(X_train[te_cols].reset_index(drop=True), y_train).astype('float32')             
                    X_val_enc = te.transform(X_val[te_cols].reset_index(drop=True)).astype('float32') 
                    X_test_enc = te.transform(X_test[te_cols].reset_index(drop=True)).astype('float32')
                
                    te_names = [f"{col}_TE_s{sm}" for col in te_cols]
                    X_train[te_names] = X_train_enc
                    X_val[te_names] = X_val_enc
                    X_test[te_names] = X_test_enc
            
                X_train = X_train.drop(self.cat_features, axis=1)
                X_val = X_val.drop(self.cat_features, axis=1)
                X_test = X_test.drop(self.cat_features, axis=1)
            
            num_features = X_train.select_dtypes(exclude=['category']).columns.tolist()
            cat_features = X_train.select_dtypes(include=['category']).columns.tolist()
            print(f'Fold {n_fold+1}')
            
            if "LGBM" in model_name:
                model.fit(X_train, y_train, eval_set=[(X_val, y_val)])
                
            elif "XGB" in model_name:
                model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
                
            elif any(model in model_name for model in ["FTT", "TabMD", 'ResNet', 'Realmlp']):
                model.fit(
                    X_train, y_train,
                    X_val, y_val,
                    cat_col_names=cat_features
                )
            
            elif any(model in model_name for model in ["NN", "TabM"]):
                model.num_features = X_train.select_dtypes(exclude=['category']).columns.tolist()
                model.cat_features = X_train.select_dtypes(include=['category']).columns.tolist()
                model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

            elif "CAT" in model_name:
                X_train = Pool(X_train, label=y_train, cat_features=cat_features)
                X_val = Pool(X_val, label=y_val, cat_features=cat_features)
                X_test = Pool(X_test, cat_features=cat_features)
                model.fit(X_train, eval_set=X_val, verbose=False)
                
            elif any(model in model_name for model in ["HGB", "YDF"]):
                model.fit(X_train, y_train, X_val=X_val, y_val=y_val)

            elif "Ensemble" in model_name:
                model.fit(X_train, y_train)
            
            else:
                encoder = FeatureEncoder(num_features=num_features, cat_features=cat_features)
                encoder.fit(X_train)
                X_train, X_val, X_test = encoder.transform_fold(X_train, X_val, X_test)          
                model.fit(X_train, y_train)

            if self.task_type == "regression":
                y_pred_val = model.predict(X_val)           
                test_pred += model.predict(X_test) / self.n_splits
            elif self.task_type == "binary":
                y_pred_val = model.predict_proba(X_val)[:, 1]            
                test_pred += model.predict_proba(X_test)[:, 1] / self.n_splits
            elif self.task_type == "multiclass":
                y_pred_val = model.predict_proba(X_val)
                test_pred += model.predict_proba(X_test) / self.n_splits
                
            oof_pred[valid_id] = y_pred_val
            score = self.metrics.score(y_val, y_pred_val)
            print(score)
            self.scores.loc[f'{model_name}', f'Fold {n_fold+1}'] = score

        self.scores.loc[f'{model_name}', 'Score'] = self.scores.loc[f'{model_name}'][1:].mean()

        return oof_pred, test_pred

    def _pred_cols(self, model_name):
        if self.task_type == "multiclass":
            return [f"{model_name}_{i}" for i in range(self.n_classes)]
        return [model_name]
        
    def _save_preds(self, model_name, oof_pred, test_pred):
        np.save(f"{model_name}_oof.npy", oof_pred)
        np.save(f"{model_name}_test.npy", test_pred)
        
    def _load_preds(self, model_name):
        oof = np.load(f"/kaggle/input/datasets/mikhailnaumov/evp-models/{model_name}_oof.npy")
        test = np.load(f"/kaggle/input/datasets/mikhailnaumov/evp-models/{model_name}_test.npy")
        return oof, test
        
    def _append_preds(self, model_name, oof_pred, test_pred):
        cols = self._pred_cols(model_name)
        self.OOF_preds = pd.concat([self.OOF_preds, pd.DataFrame(oof_pred, columns=cols)], axis=1).reset_index(drop=True)
        self.TEST_preds = pd.concat([self.TEST_preds, pd.DataFrame(test_pred, columns=cols)], axis=1).reset_index(drop=True)
                    
    def run(self):
        for model_name, model in tqdm(self.models.items()):
            if self.training:
                oof_pred, test_pred = self.train(model, self.X.copy(), self.y, self.test.copy(), model_name)
                self._save_preds(model_name, oof_pred, test_pred)
            else:
                oof_pred, test_pred = self._load_preds(model_name)
                
                for n_fold, (train_id, valid_id) in enumerate(self.folds.split(oof_pred, self.y)):
                    y_pred_val, y_val = oof_pred[valid_id], self.y[valid_id]
                    self.scores.loc[f'{model_name}', f'Fold {n_fold+1}'] = self.metrics.score(y_val, y_pred_val)
                self.scores.loc[f'{model_name}', 'Score'] = self.scores.loc[f'{model_name}'][1:].mean()
                
            self._append_preds(model_name, oof_pred, test_pred)
            
        if len(self.models)>1:
            if self.task_is_regression:
                meta_model = LinearRegression()
            elif self.task_type == "multiclass":
                meta_model = LogisticRegression(max_iter=2000, random_state=42, penalty="l2", solver="lbfgs", C=0.05, n_jobs=-1)
            else:
                meta_model = LogisticRegression(max_iter=2000, random_state=42, penalty="l2", solver="lbfgs", C=0.1)

            self.OOF_preds = self.prob_to_logit(self.OOF_preds)
            self.TEST_preds = self.prob_to_logit(self.TEST_preds)
            Ensemble_OOF, Ensemble_TEST = self.train(meta_model, self.OOF_preds, self.y, self.TEST_preds, 'Ensemble')
            
            self.scores.loc["Ensemble", 'Score'] = self.metrics.score(self.y, Ensemble_OOF)
            self.scores = self.scores.sort_values('Score')
            self.plotter.score_bar(self.scores)
            self.plotter.plot_result(self.y, Ensemble_OOF)
            return Ensemble_TEST, Ensemble_OOF
        else:
            print(Style.BRIGHT+Fore.GREEN+f'{model_name} score {self.scores.loc[f"{model_name}", "Score"]:.7f}\n')
            self.plotter.plot_result(self.y, self.OOF_preds[f'{model_name}'])
            return self.TEST_preds[f'{model_name}'], self.OOF_preds[f'{model_name}']
        
    def prob_to_logit(self, p):
        p = np.clip(p, 1e-15, 1.0 - 1e-15).astype(np.float64)
        return np.clip(np.log(p / (1.0 - p)), -30, 30).astype(np.float32)

In [ ]:
trainer = Trainer(X, y, test, models, num_features, cat_features, training=False)
TEST_preds, OOF_preds = trainer.run()

### What each plot shows

- **ROC curve (one‑vs‑rest)**  
  Shows the model’s ability to discriminate between classes across all thresholds. High AUC (~0.94 for both classes) indicates strong overall separability and that the classifier performs well at ranking positives above negatives.

- **Precision‑Recall curves (one‑vs‑rest)**  
  Illustrate the trade‑off between precision and recall for each class. Class 0 has very high average precision (AP ≈ 0.987), meaning predictions for that class are both precise and recallable; class 1 has lower AP (≈ 0.761), indicating that capturing more positives for class 1 comes at a larger precision cost.

- **Confusion matrix**  
  Summarizes absolute counts of true/false positives and negatives at the chosen decision threshold. It reveals the balance of errors: many true negatives and true positives, but also substantial false negatives and false positives. Use these counts to translate model errors into business costs.

- **Calibration curves (all classes)**  
  Compare predicted probabilities to observed outcome frequencies. Lines close to the diagonal show that predicted probabilities are well calibrated on average, so probability outputs can be interpreted reliably for scoring and thresholding; check calibration by segment for local deviations.

# <p style="border-radius: 40px; color: white; font-weight: bold; font-size: 150%; text-align: center; background-color:#3cb371; padding: 5px 5px 5px 5px;">Submission</p>

In [ ]:
class OOF_TEST_plot(Config):

    def plot(self, oof, test, y):
        if self.task_is_regression:
            return self.oof_test_regression(oof, test, y)
        return self.oof_test_classification(oof, test, y, sample_size=10000)

    def oof_test_regression(self, oof, test, y):   
        sns.set_style('whitegrid')
        colors = {'OOF':'#3cb371', 'TEST':'#ef5350', 'TRAIN':'#5c9ded'}
        
        fig, axes = plt.subplots(2, 2, figsize=(14, 14))
        ax1, ax2, ax3, ax4 = axes.ravel()
        
        # ECDF
        def plot_ecdf(ax, vals, label, color):
            s = np.sort(vals)
            y = np.arange(1, len(s)+1)/len(s)
            ax.plot(y, s, label=label, color=color, lw=1.6)
    
        plot_ecdf(ax1, oof, 'OOF', colors['OOF'])
        plot_ecdf(ax1, test, 'TEST', colors['TEST'])
        plot_ecdf(ax1, y, 'TRAIN', colors['TRAIN'])
        ax1.set_title('ECDF: OOF vs TEST')
        ax1.legend()
    
        # KDE
        sns.kdeplot(oof, ax=ax2, label='OOF', color=colors['OOF'], lw=1.6)
        sns.kdeplot(test, ax=ax2, label='TEST', color=colors['TEST'], lw=1.6)
        sns.kdeplot(y, ax=ax2, label='TRAIN', color=colors['TRAIN'], lw=1.6)
        ax2.set_title('KDE / Histogram')
        ax2.legend()
    
        # Quantile difference
        q = np.linspace(0,1,101)
        q_oof = np.quantile(oof, q)
        q_test = np.quantile(test, q)
        ax3.plot(q, q_test - q_oof, color='purple')
        ax3.axhline(0, color='k', ls='--')
        ax3.set_xlabel('Quantile')
        ax3.set_ylabel('TEST - OOF')
        ax3.set_title('Quantile difference')
    
        # Stats text
        ks_p = ks_2samp(oof, test).pvalue
        w = wasserstein_distance(oof, test)
        txt = (
            f"KS p-value: {ks_p:.3g}\n"
            f"Wasserstein: {w:.4f}\n"
            f"OOF mean/median/std: {float(oof.mean()):.4f} / {float(np.median(oof)):.4f} / {float(oof.std()):.4f}\n"
            f"TEST mean/median/std: {float(test.mean()):.4f} / {float(np.median(test)):.4f} / {float(test.std()):.4f}"
        )
        ax4.axis('off')
        ax4.text(0.01, 0.98, txt, va='top', fontsize=11, family='monospace')
    
        plt.tight_layout()
        plt.show()

    def oof_test_classification(self, oof, test, y, sample_size=10000):
        sns.set_style('whitegrid')
        colors = np.array(['#3cb371','#ef5350','#5c9ded'])

        if oof.ndim == 1:
            oof = np.column_stack([1 - oof, oof])
            test = np.column_stack([1 - test, test])
    
        n = min(sample_size, test.shape[0])
        rng = np.random.RandomState(42)
        idx = rng.choice(test.shape[0], n, replace=False)
    
        X = test[idx]
        X_pca = PCA(n_components=min(50, X.shape[1], 50), random_state=42).fit_transform(X)
        emb = umap.UMAP(n_components=2, random_state=42).fit_transform(X_pca)
    
        preds = X.argmax(axis=1)
        max_prob = X.max(axis=1)
        entropy = -(X * np.log(X + 1e-12)).sum(axis=1)
    
        fig, axes = plt.subplots(2, 2, figsize=(14, 14))
        ax1, ax2, ax3, ax4 = axes.ravel()
    
        for k in range(self.n_classes):
            mask = preds == k
            ax1.scatter(emb[mask,0], emb[mask,1], s=12, alpha=0.7, color=colors[k], label=self.labels[k])    
        ax1.set_title("UMAP of predicted probabilities (colored by class)")
        ax1.legend(title="Classes", loc='upper left', frameon=False)
        
        sc = ax2.scatter(emb[:,0], emb[:,1], c=max_prob, cmap='viridis', s=12, alpha=0.9)
        plt.colorbar(sc, ax=ax2, label='max probability')
        ax2.set_title("Confidence (max prob) on embedding")
        
        oof_classes = oof.argmax(axis=1)
        test_classes = test.argmax(axis=1)
        train_counts = pd.Series(y).value_counts(normalize=True) if y is not None else None
    
        dist_df = pd.DataFrame({
            "OOF": pd.Series(oof_classes).map(lambda x: self.labels[x]).value_counts(normalize=True),
            "TEST": pd.Series(test_classes).map(lambda x: self.labels[x]).value_counts(normalize=True),
        }).reindex(self.labels, fill_value=0).reset_index().rename(columns={"index":"Class"}).melt(id_vars="Class", var_name="Source", value_name="Proportion")
    
        train_ser = pd.Series(y).map(lambda x: self.labels[x]).value_counts(normalize=True).reindex(self.labels, fill_value=0)
        tmp = pd.DataFrame({'Class':train_ser.index, 'Source':'Train', 'Proportion':train_ser.values})
        dist_df = pd.concat([dist_df, tmp], ignore_index=True)
    
        sns.barplot(data=dist_df, x='Class', y='Proportion', hue='Source', palette=colors, ax=ax3)
        ax3.set_title('Class proportions: OOF vs TEST vs Train')
        ax3.set_ylim(0, dist_df['Proportion'].max() * 1.15)
        for p in ax3.patches:
            h = p.get_height()
            if h > 0:
                ax3.annotate(f"{h*100:.1f}%", (p.get_x()+p.get_width()/2, h), ha='center', va='bottom', fontsize=9, rotation=0)
        
        df_conf = pd.DataFrame({'max_prob': max_prob, 'pred': preds})
        for k in range(self.n_classes):
            vals = df_conf.loc[df_conf['pred']==k, 'max_prob']
            if len(vals) == 0:
                print(f"Class {k} ({self.labels[k]}): no predicted samples")
                continue
            sns.kdeplot(vals, ax=ax4, label=self.labels[k], color=colors[k], fill=False, linewidth=1.5)
        ax4.set_xlabel("Max probability")
        ax4.set_ylabel("Density")
        ax4.set_title("Max probability distribution by predicted class (KDE)")
        ax4.legend()
        plt.savefig('12.png', dpi=300, bbox_inches='tight')
        plt.show()

In [ ]:
submission = Config.submission
submission[Config.target] = TEST_preds
submission.to_csv("submission.csv", index=False)

display(submission.head())
p = OOF_TEST_plot()
p.plot(OOF_preds, TEST_preds, y)

### What the plots show

- **UMAP of predicted probabilities**  
  Points form several dense clusters with partial class separation: there are large homogeneous regions for class 0 and class 1 and intermediate zones where the classes mix.

- **Confidence (max probability) on embedding**  
  Confidence is heterogeneous across the embedding: high confidence concentrates inside dense clusters, while boundary regions and mixed clusters show noticeably lower max probabilities.

- **Class proportions: OOF vs TEST vs Train**  
  Class balance is stable across splits (Class 0 ≈ 82–84%, Class 1 ≈ 16–17%), indicating no major global prevalence drift between Train, OOF, and Test.

- **Max probability distribution by predicted class (KDE)**  
  Predicted probabilities for class 0 are tightly concentrated near 1.0; for class 1 the distribution is broader and shifted lower, meaning the model is generally more confident about class 0 predictions than about class 1.